# DLinear: Почему простота побеждает

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/14_dlinear.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast torch pandas numpy matplotlib

## Реализация DLinear на PyTorch

In [ ]:
import torch
import torch.nn as nn

class MovingAvg(nn.Module):
    """Скользящее среднее для декомпозиции"""
    def __init__(self, kernel_size, stride=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # Паддинг для сохранения длины
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        return x.permute(0, 2, 1)


class DLinearModel(nn.Module):
    def __init__(self, seq_len, pred_len, enc_in, kernel_size=25):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        
        # Декомпозиция
        self.decomposition = MovingAvg(kernel_size)
        
        # Отдельные линейные слои для тренда и сезонности
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        # x: [batch, seq_len, channels]
        trend = self.decomposition(x)
        seasonal = x - trend
        
        # Прогноз по каждой компоненте (channel-independent)
        trend_out = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)
        seasonal_out = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        
        return trend_out + seasonal_out


# Использование
model = DLinearModel(
    seq_len=336,      # входное окно
    pred_len=96,      # горизонт прогноза
    enc_in=7,         # количество переменных
    kernel_size=25    # размер ядра для декомпозиции
)

# Прогноз
x = torch.randn(32, 336, 7)  # batch=32, seq_len=336, channels=7
pred = model(x)              # [32, 96, 7]
print(f"Input shape: {x.shape}")
print(f"Output shape: {pred.shape}")

## DLinear с использованием NeuralForecast

In [ ]:
import pandas as pd
import numpy as np

# Создаём синтетические данные
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=500, freq='H')
y = 100 + np.cumsum(np.random.randn(500)) + 20 * np.sin(np.arange(500) / 24 * 2 * np.pi)

train_df = pd.DataFrame({
    'unique_id': 'series_1',
    'ds': dates,
    'y': y
})
print(train_df.head())

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear
from neuralforecast.losses.pytorch import MAE

models = [
    DLinear(
        h=96,                    # горизонт прогноза
        input_size=336,          # входное окно
        loss=MAE(),
        max_steps=1000,
        learning_rate=1e-3,
    )
]

nf = NeuralForecast(models=models, freq='H')
nf.fit(df=train_df)
forecasts = nf.predict()
print(forecasts.head())

## Визуализация

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))

# История (последние 200 точек)
history = train_df.tail(200)
ax.plot(history['ds'], history['y'], label='История', color='blue')

# Прогноз
ax.plot(forecasts['ds'], forecasts['DLinear'], 
        label='DLinear прогноз', linestyle='--', color='red')

ax.set_title('DLinear: прогноз временного ряда')
ax.set_xlabel('Время')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Визуализация декомпозиции

In [ ]:
# Демонстрация декомпозиции на скользящее среднее
from scipy.ndimage import uniform_filter1d

sample = train_df.tail(200)['y'].values
kernel_size = 25

# Тренд через скользящее среднее
trend = uniform_filter1d(sample, size=kernel_size, mode='nearest')
seasonal = sample - trend

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(sample, label='Исходный ряд', color='blue')
axes[0].set_title('Исходный ряд')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(trend, label='Тренд (MA)', color='green')
axes[1].set_title('Компонента тренда')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(seasonal, label='Сезонность', color='orange')
axes[2].set_title('Компонента сезонности')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()